# English High Court Data Ingestion Engine (Google Colab)

This notebook executes the optimized **Decoupled Producer-Consumer Pipeline** directly in **Google Colab** to process English High Court judgments.

### Performance & Ingestion Strategy:
- **English-Only Filtering**: Automatically filters out vernacular non-English translations.
- **Default Limit (1,000 PDFs)**: Processes up to **1,000 new unprocessed English PDFs** per run.
- **Instant Deduplication (0ms Skip)**: Skips any `doc_id` previously completed in `checkpoint.json` on Google Drive.
- **0% Google Drive IO Pollution**: Intermediate files are processed on local scratch (`/tmp/colab_scratch`). Only final Parquet datasets (`metadata.parquet`, `entities.parquet`, `documents_text.parquet`) and `checkpoint.json` live on Google Drive.
- **Automated 3-Minute Drive Sync**: Automatically updates `checkpoint.json` on Google Drive every 180 seconds.

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Step 2: Install Pipeline Dependencies

In [ ]:
!pip install -q pymupdf pyarrow duckdb rich spacy requests urllib3 matplotlib

## Step 3: Run English High Court Ingestion (Default Limit: 1000 New PDFs | 180s Auto-Sync)

In [ ]:
!python3 colab_runner.py \
    --drive-dir /content/drive/MyDrive/aws_indian_judgements \
    --scratch-dir /tmp/colab_scratch \
    --max-workers 8 \
    --batch-size 50 \
    --limit 1000 \
    --run-id hc-english-ingest-1000

## Step 4: Analyze Ingestion Throughput & Stagewise Latency

In [ ]:
!python3 analyze_throughput.py

## Step 5: Query Google Drive Parquet Datasets via DuckDB

In [ ]:
import duckdb
drive_parquet_dir = "/content/drive/MyDrive/aws_indian_judgements/parquet"

con = duckdb.connect()
query1 = f"SELECT COUNT(*) as doc_count, AVG(page_count) as avg_pages FROM '{drive_parquet_dir}/metadata.parquet'"
meta_df = con.execute(query1).df()
print("=== Metadata Summary ===")
print(meta_df)

query2 = f"SELECT type, COUNT(*) as frequency FROM '{drive_parquet_dir}/entities.parquet' GROUP BY type ORDER BY frequency DESC LIMIT 10"
ent_df = con.execute(query2).df()
print("\n=== Top Extracted Entity Types ===")
print(ent_df)